# Wikipedia Polls — Data Collection

## Why Wikipedia?

Wikipedia maintains a continuously updated, community-curated page that aggregates **all major nationwide polls** for the 2024 US presidential election into structured HTML tables. This makes it an ideal single source for:
- A large number of polls from many different pollsters across ~4 months
- Consistent column structure (pollster, date, sample size, candidate percentages)
- No paywall or API key required

The alternative — collecting polls from individual pollster websites — would require dozens of separate scrapers and produce inconsistent formats.

## What we collect

We focus on the **Harris vs Trump** head-to-head matchup (the main general election race), covering **July 5 – November 4, 2024** — the same window used for all other data sources in this project. For each poll we record:
- `Pollster` — the organisation that conducted the poll
- `Date` — end date of the field period
- `SampleSize` — number of respondents and voter type (LV = likely voters, RV = registered voters)
- `Trump` / `Harris` — reported support percentage
- `Margin` — Trump − Harris (positive = Trump leads, negative = Harris leads)

**Source:** [Wikipedia – Nationwide opinion polling for the 2024 United States presidential election](https://en.wikipedia.org/wiki/Nationwide_opinion_polling_for_the_2024_United_States_presidential_election)

**Output:** `Data/1_Bronze/Polls/wikipedia_polls.csv`

## 0. Setup

In [8]:
import sys, os, re, warnings
warnings.filterwarnings('ignore')

import requests
from bs4 import BeautifulSoup
import pandas as pd

BRONZE_PATH = '../../Data/1_Bronze/Polls/'
URL = (
    'https://en.wikipedia.org/wiki/'
    'Nationwide_opinion_polling_for_the_2024_United_States_presidential_election'
)
print('Setup complete.')

Setup complete.


## 1. Scraping with BeautifulSoup

### 1.1 Fetch the page

We use the `requests` library to download the raw HTML of the Wikipedia page.
A custom `User-Agent` header is set so the server can identify the request as coming from a research project
(Wikipedia blocks the default `python-requests` agent to discourage automated scraping).

`raise_for_status()` will throw an error immediately if the server returns a non-200 response
(e.g. 404 Not Found or 429 Too Many Requests), rather than silently continuing with bad data.

In [9]:
headers = {'User-Agent': 'Mozilla/5.0 (compatible; SMWA-project/1.0)'}
response = requests.get(URL, headers=headers, timeout=20)
response.raise_for_status()

print('Status :', response.status_code)
print('Size   :', f'{len(response.text):,}', 'characters')

Status : 200
Size   : 1,593,636 characters


### 1.2 Parse HTML structure with BeautifulSoup

BeautifulSoup turns the raw HTML string into a navigable tree of elements.
We use two core methods:
- `find()` — returns the **first** matching element (used for unique elements like the page title)
- `find_all()` — returns **all** matching elements (used to scan headings and locate all tables)

Wikipedia stores all its data tables with `class='wikitable'`. There are 80 of them on this page
because it covers many race configurations (Biden vs Trump, RFK Jr. scenarios, hypothetical matchups, etc.).
We need to identify only the relevant subset.

In [10]:
soup = BeautifulSoup(response.text, 'html.parser')

# Page title via find()
print('Page title:', soup.find('h1', id='firstHeading').get_text())

# First 12 section headings via find_all()
print('\nSections:')
for h in soup.find_all(['h2', 'h3'])[:12]:
    print(' •', h.get_text(strip=True))

all_tables = soup.find_all('table', class_='wikitable')
print(f'\nTotal wikitables: {len(all_tables)}')

Page title: Nationwide opinion polling for the 2024 United States presidential election

Sections:
 • Contents
 • Polling aggregation
 • Kamala Harris vs. Donald Trump
 • Kamala Harris vs. Donald Trump vs. Robert F. Kennedy Jr. vs. Jill Stein vs. Chase Oliver vs. Cornel West
 • National poll results
 • Kamala Harris vs. Donald Trump
 • Kamala Harris vs. Donald Trump vs. Robert F. Kennedy Jr. vs. Cornel West vs. Jill Stein
 • Hypothetical polling
 • Robert F. Kennedy Jr.
 • Joe Biden vs. Donald Trump
 • Undeclared and generic candidates
 • Limitations

Total wikitables: 80


### 1.3 Select relevant tables and convert to DataFrames

**Table selection strategy:** We keep only tables whose header text contains both `trump` and `harris`/`kamala`,
and that have more than 10 rows (to skip small summary tables). This filters the 80 wikitables down to
the 6 that cover the main head-to-head matchup.

**Why `pd.read_html(str(t))`?** BeautifulSoup gives us the HTML element; `pd.read_html()` converts it to a
DataFrame automatically. We pass `str(t)` (the HTML of just that one table) rather than the full page URL,
so pandas only parses the table we already selected.

**MultiIndex flattening:** Wikipedia uses `rowspan` and `colspan` in its table headers, which `pd.read_html()`
translates into a MultiIndex. We flatten this to simple strings so column names are easy to work with.

In [11]:
def col_text(table):
    """Return all header text of a table as a single lowercase string."""
    return ' '.join(th.get_text(strip=True) for th in table.find_all('th')).lower()

# Keep Harris vs Trump tables with enough rows
poll_tables = [
    t for t in all_tables
    if 'trump' in col_text(t)
    and ('harris' in col_text(t) or 'kamala' in col_text(t))
    and len(t.find_all('tr')) > 10
]
print(f'Polling tables selected: {len(poll_tables)}')

# Convert each BS4 table element to DataFrame
raw_parts = []
for t in poll_tables:
    for d in pd.read_html(str(t)):
        # Flatten MultiIndex columns from Wikipedia rowspan/colspan headers
        if isinstance(d.columns, pd.MultiIndex):
            d.columns = [
                ' '.join(str(x) for x in col if str(x) != 'nan').strip()
                for col in d.columns
            ]
        raw_parts.append(d)

df_raw = pd.concat(raw_parts, ignore_index=True)
print(f'Raw shape: {df_raw.shape}')
print('Raw columns:', df_raw.columns.tolist())

Polling tables selected: 6
Raw shape: (342, 11)
Raw columns: ['Poll source', 'Date', 'Sample size[c]', 'Margin of error', 'Kamala Harris Democratic', 'Donald Trump Republican', 'Others/ Undecided', 'Lead', 'Robert F. Kennedy Jr. Independent', 'Cornel West Independent', 'Jill Stein Green']


## 2. Data Cleaning

The raw data has several issues that need fixing before analysis:

1. **Column names** — Wikipedia uses long descriptive names like `'Donald Trump Republican'` and `'Kamala Harris Democratic'`. We rename these to consistent short names.
2. **Percentage strings** — Values are stored as strings like `'47%'` or `'—N/a'`. We strip `%`, remove non-numeric characters, and convert to float.
3. **Date ranges** — Poll dates are given as ranges like `'October 1–5, 2024'`. We extract the **end date** (when polling stopped) as the representative date, since that is closest to when the data was published.
4. **Footnotes** — Wikipedia adds citation footnotes in brackets like `[9]` to pollster names and dates. These need to be stripped.
5. **Margin** — We compute `Trump − Harris` ourselves (as a signed value) rather than using Wikipedia's `Lead` column, which stores absolute values.

In [12]:
# Rename columns to a consistent schema
rename = {}
for col in df_raw.columns:
    low = col.lower()
    if ('trump' in low or 'republican' in low) and 'harris' not in low and 'democrat' not in low:
        rename[col] = 'Trump'
    elif 'harris' in low or 'kamala' in low or 'democrat' in low:
        rename[col] = 'Harris'
    elif ('poll' in low or 'source' in low or 'firm' in low) and 'date' not in low:
        rename[col] = 'Pollster'
    elif 'date' in low or 'field' in low:
        rename[col] = 'Date'
    elif 'sample' in low or low.startswith('n '):
        rename[col] = 'SampleSize'

df = df_raw.rename(columns=rename)
df = df.loc[:, ~df.columns.duplicated(keep='first')]

keep = [c for c in ['Pollster', 'Date', 'SampleSize', 'Trump', 'Harris'] if c in df.columns]
df = df[keep]
print('Columns kept:', df.columns.tolist())
df.head(3)

Columns kept: ['Pollster', 'Date', 'SampleSize', 'Trump', 'Harris']


,Pollster,Date,SampleSize,Trump,Harris
0,Zogby[9],"November 2–3, 2024","1,005 (LV)",46%,49%
1,Research Co.[10],"November 2–3, 2024","1,003 (LV)",46%,48%
2,Reuters/Ipsos[11],"November 1–3, 2024",973 (LV),48%,50%


In [13]:
# Parse percentages
for col in ['Trump', 'Harris']:
    if col in df.columns:
        df[col] = (
            df[col].astype(str)
                   .str.replace('%', '', regex=False)
                   .str.replace('[^0-9.\-]', '', regex=True)
                   .pipe(pd.to_numeric, errors='coerce')
        )

# Parse dates — handle ranges like "Oct 1–5, 2024" by taking the end date
def parse_date(s):
    s = re.sub(r'\[.*?\]', '', str(s)).strip()   # strip footnotes
    for sep in ['\u2013', '\u2014', '\u2012']:    # en-dash, em-dash, figure-dash
        if sep in s:
            left, right = s.split(sep, 1)
            right = right.strip()
            # If the right side has no month name, prepend it from the left side
            if not re.search(r'[A-Za-z]', right.split(',')[0]):
                month = re.match(r'[A-Za-z]+', left.strip())
                if month:
                    right = month.group() + ' ' + right
            s = right
            break
    try:
        return pd.to_datetime(s.strip())
    except Exception:
        return pd.NaT

df['Date'] = df['Date'].apply(parse_date)

# Keep only polls from July 5, 2024 onward (to match other data sources)
df = df[df['Date'] >= '2024-07-05']
df = df.dropna(subset=['Trump', 'Harris'], how='all').reset_index(drop=True)
df = df.sort_values('Date').reset_index(drop=True)

# Compute signed margin (positive = Trump leads)
df['Margin'] = df['Trump'] - df['Harris']

print(f'Polls loaded : {len(df)}')
print(f'Date range   : {df["Date"].min().date()} → {df["Date"].max().date()}')
df.head(5)

Polls loaded : 249
Date range   : 2024-07-06 → 2024-11-04


,Pollster,Date,SampleSize,Trump,Harris,Margin
0,Bendixen & Amandi International (D)[201],2024-07-06,"1,000 (LV)",41.0,42.0,-1.0
1,Emerson College[200],2024-07-08,"1,370 (RV)",49.0,43.0,6.0
2,NBC News[198],2024-07-09,800 (RV),47.0,45.0,2.0
3,ABC News/The Washington Post/Ipsos[199],2024-07-09,"2,041 (RV)",47.0,49.0,-2.0
4,The Economist/YouGov[303],2024-07-09,"1,443 (RV)",42.0,38.0,4.0


## 3. Save to Bronze

We save the cleaned DataFrame as a CSV to the Bronze layer. The Bronze layer stores
**raw collected data** with minimal transformation — only cleaning needed to make the data
machine-readable (type conversion, column renaming). No aggregation or feature engineering
happens here; that is done in the analysis notebook.

In [14]:
os.makedirs(BRONZE_PATH, exist_ok=True)
out_path = os.path.join(BRONZE_PATH, 'wikipedia_polls.csv')
df.to_csv(out_path, index=False)
print(f'Saved {len(df)} rows → {out_path}')
print(df.dtypes)

Saved 249 rows → ../../Data/1_Bronze/Polls/wikipedia_polls.csv
Pollster              object
Date          datetime64[ns]
SampleSize            object
Trump                float64
Harris               float64
Margin               float64
dtype: object
